## 2. Arnoldi: an orthonormal Krylov basis (`gmres_v1`)

*Same math, computed better.* This step changes how GMRES computes its answer, not what the answer is. In exact arithmetic the new solver produces exactly the same iterates as `gmres_v0`. In floating point it should agree with `gmres_v0` up to round-off wherever `gmres_v0` still works, and we'll check that it does.

**Where Step 1 left off.** We want to solve $Ax = b$, where $A$ is a nonsingular $n \times n$ matrix that we can only apply to vectors, through a function `matvec` $: v \mapsto Av$. Starting from $x_0 = 0$, GMRES picks at step $k$ the vector $x_k$ in the Krylov space

$$
\mathcal{K}_k = \operatorname{span}\{\, b,\ Ab,\ \dots,\ A^{k-1} b \,\}
$$

whose residual $r_k = b - A x_k$ has the smallest 2-norm, and stops once $\|r_k\|_2 \le \text{rtol} \cdot \|b\|_2$. `gmres_v0` did this literally. It put the Krylov vectors in the columns of the $n \times k$ matrix $K_k = [\, b \mid Ab \mid \cdots \mid A^{k-1} b \,]$, wrote $x_k = K_k c$ with a weight vector $c \in \mathbb{R}^k$, and solved the least-squares problem $\min_c \|b - (A K_k)\, c\|_2$.

On `diag_spread`, the 20×20 diagonal matrix with eigenvalues from 1 to $10^4$ and $b = (1, \dots, 1)$, that broke. The residual went *up* from step 5 on and ended near 0.95 instead of 0. The columns of $K_k$ all turn toward the same direction, so the condition number $\operatorname{cond}(K_k)$ passes $1/\varepsilon_{\text{machine}} \approx 4.5 \times 10^{15}$ at $k = 5$, and from then on the least-squares solve is mostly round-off. ($\varepsilon_{\text{machine}} \approx 2.2 \times 10^{-16}$ is the gap between 1 and the next double-precision number.)

**The plan for this step.** `gmres_v1` keeps everything about `gmres_v0` except the basis. Instead of $b, Ab, A^2 b, \dots$ it builds an *orthonormal* basis of the same space, one vector per matvec. This is the **Arnoldi process**.

To make each new vector perpendicular to the earlier ones, Arnoldi computes and subtracts its components along them. `gmres_v1` uses those numbers only to subtract. Step 3 shows that they also make the least-squares problem much smaller.

The step also checks that the basis we actually compute, in floating point, is as orthonormal as section 2.1 promises. That check decides how the subtraction is done.

### Setup

[tutorial_utils.py](tutorial_utils.py) holds what every step of the tutorial uses: `CountingMatvec` from Step 1, the example problems and the plot style. Each example builder returns a pair `(A, b)`. As in Step 1, a solver only ever gets the matrix wrapped in a `CountingMatvec`; we use `A` directly only to *check* answers.

The example problems in this notebook, each a matrix $A$ with a right-hand side $b$:

- **`geo3`**: the 3×3 running example from Step 1,
  $$
  A = \begin{pmatrix} 3 & 1 & 0 \\ 0 & 2 & 1 \\ 1 & 0 & 2 \end{pmatrix}, \qquad b = (1, 1, 1) .
  $$
  Nonsymmetric and well conditioned, and small enough to check every number by hand.
- **`diag_narrow`**: the 100×100 diagonal matrix $A = \operatorname{diag}(\lambda_1, \dots, \lambda_{100})$ with eigenvalues evenly spaced from 1 to 1.5, $\lambda_i = 1 + 0.5\,(i-1)/99$, and $b = (1, \dots, 1)$. `gmres_v0` solved it in 11 steps.
- **`diag_spread`**: the 20×20 diagonal matrix $A = \operatorname{diag}(\lambda_1, \dots, \lambda_{20})$ with eigenvalues evenly spaced on a log scale from 1 to $10^4$, $\lambda_i = 10^{4(i-1)/19}$, and $b = (1, \dots, 1)$. `gmres_v0` failed on it.
- **`diag_spread(20, decades=12)`** (new in this step): the same kind of 20×20 diagonal matrix with the eigenvalues spread over twelve decades instead of four, $\lambda_i = 10^{12(i-1)/19}$ from 1 to $10^{12}$, and $b = (1, \dots, 1)$, so $\operatorname{cond}(A) = 10^{12}$. It is used only in section 2.4, where it makes the round-off of orthogonalisation large enough to see.

`%autoreload 2` picks up edits to `tutorial_utils.py` without restarting the kernel.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np  # noqa: I001
import matplotlib.pyplot as plt

from tutorial_utils import (
    # the matvec wrapper from Step 1
    CountingMatvec,
    # loading definitions from an earlier step's notebook
    import_from_notebook,
    # example problems; each returns (A, b)
    geo3, diag_narrow, diag_spread,
    # checking one solver version against another
    compare_versions,
    # plot style
    style_axes, plot_residuals, add_figure_caption, LINE_COLOR, REFERENCE_COLOR, SERIES_COLORS,
)

np.set_printoptions(precision=4, suppress=True)

A_geo3, b_geo3 = geo3()              # 3 x 3, the running example from Step 1
A_narrow, b_narrow = diag_narrow()   # 100 x 100 diagonal, eigenvalues in [1, 1.5]
A_spread, b_spread = diag_spread()   # 20 x 20 diagonal, eigenvalues from 1 to 1e4
A_spread_12, b_spread_12 = diag_spread(20, decades=12)   # 20 x 20 diagonal, eigenvalues from 1 to 1e12

**From Step 1.** Two functions built in Step 1 are needed again here: `krylov_matrix`, which builds the literal basis $K_k$, and `gmres_v0`, the solver this step improves. Instead of copying them, `import_from_notebook` loads them straight from [GMRES_1_krylov.ipynb](GMRES_1_krylov.ipynb). It runs only that notebook's imports and function definitions, not its plots or examples. To read either function here, run `gmres_v0??` or `krylov_matrix??`.

In [ ]:
# Built in Step 1. Loaded from that notebook, so it stays the only copy.
krylov_matrix, gmres_v0 = import_from_notebook("GMRES_1_krylov.ipynb", "krylov_matrix", "gmres_v0")

In [ ]:
# Step 1's failure, once more: gmres_v0 on diag_spread.
x_v0, record_v0 = gmres_v0(CountingMatvec(A_spread), b_spread)
relative_v0 = np.array(record_v0["residual_history"]) / np.linalg.norm(b_spread)

print("converged:", record_v0["converged"], "  iterations:", record_v0["iterations"])
print(f"||r_20|| / ||b|| = {relative_v0[-1]:.4f}")
print("first step where the residual went up:",
      next(k for k in range(1, len(relative_v0)) if relative_v0[k] > relative_v0[k - 1]))

### 2.1 Same space, better basis

Step 1 ended on this point: the search space $\mathcal{K}_k$ is fine, and the problem is the basis we used to describe it. First, why changing the basis can't change the answer.

Let $W$ be *any* $n \times k$ matrix whose columns are a basis of $\mathcal{K}_k$. Every vector of $\mathcal{K}_k$ is $W z$ for exactly one $z \in \mathbb{R}^k$, so

$$
\min_{x \in \mathcal{K}_k} \|b - A x\|_2 \;=\; \min_{z \in \mathbb{R}^k} \big\|b - (A W)\, z\big\|_2 .
$$

Both sides search the same set of vectors, so they find the same minimiser $x_k$. It is unique: $A$ is nonsingular, so the columns of $AW$ are independent like those of $W$, and a least-squares problem with independent columns has exactly one solution. $K_k$ is one choice of $W$. **Any other basis gives the same $x_k$ in exact arithmetic.** What the choice does affect is how much round-off the least-squares solve picks up, and that is governed by $\operatorname{cond}(W)$.

The best-conditioned basis is an **orthonormal** one: vectors $v_1, \dots, v_k$ with

$$
v_i^\mathsf{T} v_j = \delta_{ij} =
\begin{cases} 1 & i = j, \\ 0 & i \ne j. \end{cases}
$$

In words: each has length 1, and each pair is perpendicular. Put them in the columns of $V_k = [\, v_1 \mid \cdots \mid v_k \,]$. Then $V_k^\mathsf{T} V_k = I$, and $\operatorname{cond}(V_k) = 1$, the smallest a condition number can be.

### 2.2 Building it from matvecs: the Arnoldi process

We can't build $b, Ab, A^2 b, \dots$ first and orthonormalise afterwards. For `diag_spread`, entry $i$ of $A^j b$ is $\lambda_i^j$, where $\lambda_i$ is the $i$-th eigenvalue. These range from 1 to $10^{4j}$, so by $A^4 b$ the entries belonging to the small eigenvalues are below round-off compared with the large ones. They are lost before any orthogonalisation could recover them. Instead we orthonormalise **as we go**, and apply $A$ to the newest *orthonormal* vector rather than to the newest Krylov vector.

Start with $\beta = \|b\|_2$ and $v_1 = b / \beta$, which spans $\mathcal{K}_1$. Suppose $v_1, \dots, v_j$ are an orthonormal basis of $\mathcal{K}_j$. Then step $j$ is:

1. **One matvec:** $w = A v_j$. Since $v_j$ is a polynomial in $A$ of degree less than $j$ applied to $b$ (Step 1.2), $A v_j$ is one of degree less than $j + 1$, so $A v_j \in \mathcal{K}_{j+1}$.
2. **Remove the old directions.** Compute the components of $w$ along $v_1, \dots, v_j$ and subtract them all:
   $$
   h_{ij} = v_i^\mathsf{T} w \quad (i = 1, \dots, j), \qquad w \leftarrow w - \sum_{i=1}^{j} h_{ij}\, v_i .
   $$
   $h_{ij}$ is the component of $w$ along $v_i$. Afterwards $w$ is perpendicular to all of $v_1, \dots, v_j$: for each $l \le j$, $v_l^\mathsf{T} \big(w - \sum_i h_{ij} v_i\big) = h_{lj} - \sum_i h_{ij}\, v_l^\mathsf{T} v_i = h_{lj} - h_{lj} = 0$, since $v_l^\mathsf{T} v_i$ is 1 for $i = l$ and 0 otherwise.
3. **Normalise:** $h_{j+1,j} = \|w\|_2$ and $v_{j+1} = w / h_{j+1,j}$.

The new $v_{j+1}$ is in $\mathcal{K}_{j+1}$, has length 1, and is perpendicular to $v_1, \dots, v_j$, so $v_1, \dots, v_{j+1}$ is an orthonormal basis of $\mathcal{K}_{j+1}$. If $w$ comes out as zero, $A v_j$ has no new direction to add and $\mathcal{K}_j$ has stopped growing. That is called a *breakdown*, and Step 6 is about it.

Removing components like this is **Gram–Schmidt orthogonalisation**. In exact arithmetic the order of the subtractions doesn't matter. Subtracting the component along $v_1$ first leaves the component along $v_2$ unchanged, because the part removed lies along $v_1$, which is perpendicular to $v_2$. So it makes no difference whether every $h_{ij}$ is computed from the original $A v_j$, or each one from a $w$ already reduced by the subtractions before it. In floating point the order does matter. Section 2.4 looks at what it changes, and at why the subtraction ends up being done twice.

`gmres_v1` (section 2.5) uses the $h_{ij}$ only to subtract. Section 2.4 packages one Arnoldi step as a function that also keeps them, and Step 3 puts them to use.

### 2.3 Two Arnoldi steps by hand

Back to `geo3`, the 3×3 example from Step 1:

$$
A = \begin{pmatrix} 3 & 1 & 0 \\ 0 & 2 & 1 \\ 1 & 0 & 2 \end{pmatrix}, \qquad b = (1, 1, 1) .
$$

Here $\beta = \|b\|_2 = \sqrt 3$, so $v_1 = (1, 1, 1) / \sqrt 3$. The first Arnoldi step:

In [ ]:
matvec = CountingMatvec(A_geo3)
b = b_geo3

beta = np.linalg.norm(b)
v1 = b / beta

# ----- Arnoldi step j = 1 -----
w = matvec(v1)            # A v_1   (1 matvec)
h11 = v1 @ w              # the component of w along v_1
w = w - h11 * v1          # now perpendicular to v_1
h21 = np.linalg.norm(w)
v2 = w / h21

print("beta =", beta)
print("v1   =", v1)
print("h11  =", h11, "   h21 =", h21)
print("v2   =", v2)
print("v1 . v2 =", v1 @ v2, "   ||v2|| =", np.linalg.norm(v2))
print("matvecs used so far:", matvec.count)

These can be checked by hand. $h_{11} = v_1^\mathsf{T} A v_1 = \tfrac13\, (1,1,1)\, A\, (1,1,1)^\mathsf{T}$ is a third of the sum of all the entries of $A$, which is $10/3$. Then

$$
w = A v_1 - \tfrac{10}{3} v_1 = \tfrac{1}{\sqrt 3}\Big( (4, 3, 3) - \tfrac{10}{3} (1, 1, 1) \Big) = \tfrac{1}{3\sqrt 3}\, (2, -1, -1),
$$

which has length $\sqrt 2 / 3 \approx 0.4714$, so $v_2 = (2, -1, -1) / \sqrt 6$.

The second step does the same with $A v_2$, now removing two components. Both are computed from the same $A v_2$, then both are subtracted. The last line prints what is left of $w$ along $v_1$ and $v_2$ after that, which is what a second pass of the same subtraction would remove:

In [ ]:
# ----- Arnoldi step j = 2 -----
w = matvec(v2)               # A v_2   (1 more matvec)
h12 = v1 @ w                 # both components from the same A v_2,
h22 = v2 @ w
w = w - h12 * v1 - h22 * v2  # then both subtracted
print("left along v1 and v2:  v1 . w =", v1 @ w, "   v2 . w =", v2 @ w)
h32 = np.linalg.norm(w)
v3 = w / h32

print("h12 =", h12, "   h22 =", h22, "   h32 =", h32)
print("v3  =", v3)

V_3 = np.column_stack([v1, v2, v3])
print("\nV_3^T V_3 =\n", V_3.T @ V_3)


# span{v1, v2} should be the plane spanned by b and A b. V_2 V_2^T is the
# orthogonal projection onto span{v1, v2}, so this is the part of [b, A b] outside it.
K_2 = krylov_matrix(CountingMatvec(A_geo3), b_geo3, 2)
V_2 = V_3[:, :2]
print("part of [b, A b] outside span{v1, v2}:", np.linalg.norm(K_2 - V_2 @ (V_2.T @ K_2)))
print("matvecs used so far:", matvec.count)

$V_3^\mathsf{T} V_3$ is the identity to round-off, so $v_1, v_2, v_3$ are orthonormal, and $v_1, v_2$ span the same plane as $b, Ab$. That took two matvecs, the same as building $b, Ab, A^2 b$ in Step 1.

Would a second pass find anything left to remove? In exact arithmetic, no. After the subtraction, for $i = 1, 2$,

$$
v_i^\mathsf{T} w = v_i^\mathsf{T} A v_2 - h_{12}\, v_i^\mathsf{T} v_1 - h_{22}\, v_i^\mathsf{T} v_2 = h_{i2} - h_{i2} = 0 ,
$$

because $v_i^\mathsf{T} v_1$ and $v_i^\mathsf{T} v_2$ are 1 for the matching vector and 0 for the other. The computer printed $2.3 \times 10^{-15}$ and $3.1 \times 10^{-16}$ instead. Section 2.4 shows why that matters, and why anyone would run a second pass.

### 2.4 Gram–Schmidt in floating point

Section 2.1 promised an orthonormal basis: $V_k^\mathsf{T} V_k = I$ and $\operatorname{cond}(V_k) = 1$. That holds in exact arithmetic. In floating point every dot product and every subtraction is rounded, and 2.3 already found a remainder that was not quite perpendicular to $v_1$ and $v_2$. This section checks the promise for the $V_k$ we actually compute, over many steps, and settles how the tutorial does the subtraction.

To run many steps we package one Arnoldi step as a function that keeps the $h_{ij}$, the way production does. Two conventions:

- **Indices start at 0 in code.** Column `V[:, j]` holds $v_{j+1}$, and entry `H[i, j]` holds $h_{i+1,\,j+1}$. So a step function `step(matvec, V, j)` applies $A$ to `V[:, j]` and fills `V[:, j + 1]`.
- **The arrays are allocated once, full of zeros**, with room for $m$ steps: `V` is $n \times (m+1)$ and `H` is $(m+1) \times m$. Each step fills one column of each, in place.

Column $j$ of `H` holds the numbers that Arnoldi step $j$ computes: the components $h_{1j}, \dots, h_{jj}$ it removed, then the length $h_{j+1,j}$ it divided by. Step $j$ computes nothing else, so the entries below $h_{j+1,j}$ stay zero. After $k$ steps the filled part of `H` is the $(k+1) \times k$ matrix

$$
\bar H_k =
\begin{pmatrix}
h_{11} & h_{12} & \cdots & h_{1k} \\
h_{21} & h_{22} & \cdots & h_{2k} \\
0      & h_{32} & \cdots & h_{3k} \\
\vdots & \ddots & \ddots & \vdots \\
0      & \cdots & 0      & h_{k+1,k}
\end{pmatrix} .
$$

For now it is only a record of what Arnoldi computed. Step 3 shows what it means.

There are three ways to do the subtraction, each a step function with the same arguments and results:

- **Classical Gram–Schmidt**, `arnoldi_step_classical_gram_schmidt`: compute every component $h_{ij} = v_i^\mathsf{T} w$ from the same $w = A v_j$, then subtract them all at once. This is what 2.2 wrote down and 2.3 did by hand. In code it is two products with the basis, `V.T @ w` and `V @ h`.
- **Modified Gram–Schmidt**, `arnoldi_step_modified_gram_schmidt`: remove the components one at a time, computing each $h_{ij}$ from the $w$ left over by the subtraction before it.
- **Classical Gram–Schmidt applied twice**, `arnoldi_step_classical_gram_schmidt_twice`: one classical pass, then the same two products once more on the remainder. The second pass's components are added to the first's, so `h` still records exactly what was subtracted from $A v_j$. In exact arithmetic the second pass removes nothing (2.3).

All three end the same way, in `normalise_remainder`: a breakdown test, then $h_{j+1,j} = \|w\|_2$ and $v_{j+1} = w / h_{j+1,j}$. The test asks whether the remainder is negligible, $\|w\|_2 \le \varepsilon_{\text{machine}} \cdot \|A v_j\|_2$. If it is, there is no new direction to add: the step returns `breakdown = True` and leaves $h_{j+1,j}$ and $v_{j+1}$ as zeros. The test is written `not (‖w‖ > ε‖A v_j‖)` so that a NaN also counts. Step 6 explains both choices.

`arnoldi(matvec, b, k, step=...)` runs $k$ steps from $v_1 = b / \beta$ and returns $V_{k+1} = [\, v_1 \mid \cdots \mid v_{k+1} \,]$, $\bar H_k$ and $\beta$. It's for looking at the basis; the solvers run their own loops. Its `step` argument selects one of the three, and the default is classical Gram–Schmidt applied twice, for reasons this section will show.

In [ ]:
def normalise_remainder(V, j, w, h, norm_before):
    """Finish an Arnoldi step: test for breakdown, then store the new length and basis vector.

    Shared by the three step functions below, which differ only in how they
    remove the components of A V[:, j] along V[:, 0], ..., V[:, j].

    Args:
        V: basis array, shape (n, m + 1). Column j + 1 is overwritten in place.
        j: index of the newest filled column.
        w: the remainder, A V[:, j] with its components along the basis removed.
        h: column j of H-bar with entries 0, ..., j filled; entry j + 1 is set here.
        norm_before: ||A V[:, j]||, the length before anything was removed.

    Returns:
        h: column j of H-bar, shape (m + 1,); zero below entry j + 1.
        breakdown: whether the remainder was negligible, ||w|| <= eps * norm_before.
            Column j + 1 of V is then left as zeros, and so is h[j + 1].
    """
    norm_after = np.linalg.norm(w)
    # Written with `not` so that a NaN norm also counts as a breakdown (Step 6).
    breakdown = not (norm_after > np.finfo(float).eps * norm_before)
    if breakdown:
        V[:, j + 1] = 0.0
    else:
        h[j + 1] = norm_after
        V[:, j + 1] = w / norm_after
    return h, breakdown


def arnoldi_step_classical_gram_schmidt(matvec, V, j):
    """One Arnoldi step by classical Gram-Schmidt, one pass: fill column j + 1 of V from A V[:, j].

    Every component h_ij is computed from the same A v_j, then all of them are
    subtracted at once: two products with the basis, V[:, :j+1].T @ w and
    V[:, :j+1] @ h. Columns 0, ..., j of V must be orthonormal. Column j + 1 is
    overwritten in place. In the notes' 1-based notation, V[:, j] is v_{j+1}.

    Args:
        matvec: function v -> A v.
        V: basis array, shape (n, m + 1) with m > j.
        j: index of the newest filled column.

    Returns:
        h: column j of H-bar, shape (m + 1,); zero below entry j + 1.
        breakdown: whether the remainder was negligible. Column j + 1 of V is
            then left as zeros, and so is h[j + 1].
    """
    w = matvec(V[:, j])
    norm_before = np.linalg.norm(w)
    h = np.zeros(V.shape[1])
    basis = V[:, : j + 1]

    h[: j + 1] = basis.T @ w      # every component from the same w = A v_j,
    w = w - basis @ h[: j + 1]    # then all of them subtracted at once
    return normalise_remainder(V, j, w, h, norm_before)


def arnoldi_step_modified_gram_schmidt(matvec, V, j):
    """One Arnoldi step by modified Gram-Schmidt: the components are removed one at a time.

    Same arguments and results as arnoldi_step_classical_gram_schmidt. Each dot
    product uses the w left over by the previous subtraction, so the h_ij are the
    same numbers as classical Gram-Schmidt's in exact arithmetic, with different
    round-off.
    """
    w = matvec(V[:, j])
    norm_before = np.linalg.norm(w)
    h = np.zeros(V.shape[1])

    # Remove the component along each earlier basis vector, one at a time.
    # Each dot product uses the w left over by the previous subtraction.
    for i in range(j + 1):
        h[i] = V[:, i] @ w
        w = w - h[i] * V[:, i]
    return normalise_remainder(V, j, w, h, norm_before)


def arnoldi_step_classical_gram_schmidt_twice(matvec, V, j):
    """One Arnoldi step by classical Gram-Schmidt applied twice: what production does.

    Same arguments and results as arnoldi_step_classical_gram_schmidt. The
    one-pass step, then the same two products once more on the remainder. The
    second pass's components are added to h, so column j of H-bar still records
    exactly what was subtracted from A v_j. Section 2.4 shows why the second
    pass is needed, and why two are enough.
    """
    w = matvec(V[:, j])
    norm_before = np.linalg.norm(w)
    h = np.zeros(V.shape[1])
    basis = V[:, : j + 1]

    for _ in range(2):
        components = basis.T @ w      # all from the same w, as in one classical pass
        h[: j + 1] += components
        w = w - basis @ components
    return normalise_remainder(V, j, w, h, norm_before)


def arnoldi(matvec, b, k, *, step=arnoldi_step_classical_gram_schmidt_twice):
    """Run k Arnoldi steps from b and return the basis and the Hessenberg matrix.

    For looking at V and H-bar; the solvers run their own loops. At a breakdown
    it stops early, so the arrays can have fewer columns than asked for.

    Args:
        matvec: function v -> A v.
        b: starting vector, shape (n,).
        k: number of steps.
        step: the orthogonalisation step, called as step(matvec, V, j). The
            default is the one the tutorial and production use.

    Returns:
        V: [v_1, ..., v_{k+1}], shape (n, k + 1).
        H: H-bar_k, shape (k + 1, k), with A V[:, :k] = V H.
        beta: ||b||.
    """
    n = b.shape[0]
    beta = np.linalg.norm(b)
    V = np.zeros((n, k + 1))
    H = np.zeros((k + 1, k))
    V[:, 0] = b / beta
    for j in range(k):
        H[:, j], breakdown = step(matvec, V, j)
        if breakdown:
            return V[:, : j + 2], H[: j + 2, : j + 1], beta
    return V, H, beta

First, the default step against the by-hand numbers of 2.3 on `geo3`, and one step further, to $k = 3 = n$:

In [ ]:
# The by-hand numbers from 2.3, in the layout of H-bar_2.
H_bar_2 = np.array([
    [h11, h12],
    [h21, h22],
    [0.0, h32],
])

matvec = CountingMatvec(A_geo3)
V, H, beta = arnoldi(matvec, b_geo3, 2)   # the default step: classical Gram-Schmidt applied twice
print("H-bar_2 =\n", H)
print("matvecs:", matvec.count)
print("largest difference from the by-hand V_3:     ", np.max(np.abs(V - V_3)))
print("largest difference from the by-hand H-bar_2: ", np.max(np.abs(H - H_bar_2)))

# One more step: k = 3 = n.
V, H, beta = arnoldi(CountingMatvec(A_geo3), b_geo3, 3)
print("\nH-bar_3 =\n", H)
print(f"h_13 = {H[0, 2]:.1e}    h_43 = {H[3, 2]}")

# Why h_43 is exactly 0. Run the third step again on the same basis to see its flag,
# and recompute its remainder by hand against the breakdown threshold (A used for checking only).
_, breakdown = arnoldi_step_classical_gram_schmidt_twice(CountingMatvec(A_geo3), V.copy(), 2)
w = A_geo3 @ V[:, 2]                           # A v_3
threshold = np.finfo(float).eps * np.linalg.norm(w)
for _ in range(2):
    w = w - V[:, :3] @ (V[:, :3].T @ w)
print(f"breakdown flag: {breakdown}    remainder ||w|| = {np.linalg.norm(w):.1e}    "
      f"eps * ||A v_3|| = {threshold:.1e}")

`arnoldi` reproduces the by-hand $V_3$ and $\bar H_2$ to round-off, to $1.8 \times 10^{-15}$ and $3.3 \times 10^{-15}$, with the same 2 matvecs. The difference is the second pass, which 2.3 didn't do: it moves each number by about $10^{-15}$ and changes nothing to four digits. Two entries of $\bar H_3$ print as 0.

- **$h_{43}$ is exactly 0.** At $k = 3 = n$ the Krylov space is all of $\mathbb{R}^3$, so in exact arithmetic nothing is left of $A v_3$ after its components along $v_1, v_2, v_3$ are removed. In floating point the remainder after two passes is $2.0 \times 10^{-31}$, far below the threshold $\varepsilon_{\text{machine}} \|A v_3\|_2 = 3.8 \times 10^{-16}$. So the step is flagged as a breakdown, and $h_{43}$ and $v_4$ are left as zeros. Step 6 explains the test and what a solver should do after it.
- **$h_{13} = 1.9 \times 10^{-16}$** is zero in exact arithmetic too, but only by an accident of this $A$ and $b$. Step 3 explains it.

**How orthonormal is $V_k$?** We measure it with $\|I - V_k^\mathsf{T} V_k\|_2$. For an orthonormal $V_k$ the matrix $I - V_k^\mathsf{T} V_k$ is exactly zero. Here $\|M\|_2$ is the matrix 2-norm, the largest factor by which $M$ stretches any vector.

Figure 2.1 plots it against $k$ for the three step functions, on two matrices: `diag_spread`, with eigenvalues over 4 decades (left), and `diag_spread(20, decades=12)`, over 12 (right). Both run all $n = 20$ steps.

In [ ]:
eps = np.finfo(float).eps
n_spread = b_spread.shape[0]
steps = np.arange(1, n_spread + 1)
methods = {   # label -> (step function, color, marker)
    "classical Gram–Schmidt, one pass": (arnoldi_step_classical_gram_schmidt, SERIES_COLORS[1], "s"),
    "modified Gram–Schmidt": (arnoldi_step_modified_gram_schmidt, SERIES_COLORS[2], "^"),
    "classical Gram–Schmidt applied twice": (arnoldi_step_classical_gram_schmidt_twice, SERIES_COLORS[0], "o"),
}
runs_12 = {}  # label -> (V, H) on the 12-decade matrix, for the table further down

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), sharey=True, constrained_layout=True)
for ax, (title, A, b) in zip(axes, [("diag_spread, eigenvalues over 4 decades", A_spread, b_spread),
                                     ("diag_spread(20, decades=12)", A_spread_12, b_spread_12)]):
    print(f"{title}:  ||I - V_k^T V_k||_2 at k = 5, 12, 16, 18, 20, and cond(V_20)")
    for label, (step, color, marker) in methods.items():
        V, H, _ = arnoldi(CountingMatvec(A), b, n_spread, step=step)
        if A is A_spread_12:
            runs_12[label] = (V, H)
        loss = [np.linalg.norm(np.eye(k) - V[:, :k].T @ V[:, :k], 2) for k in steps]
        ax.semilogy(steps, loss, color=color, linewidth=2, marker=marker, markersize=5, label=label)
        print(f"  {label:38s}" + "  ".join(f"{loss[k - 1]:.1e}" for k in (5, 12, 16, 18, 20))
              + f"    cond(V_20) = {np.linalg.cond(V[:, :n_spread]):.10f}")
    ax.axhline(eps, color=REFERENCE_COLOR, linewidth=1)
    ax.text(n_spread, eps / 1.6, r"$\varepsilon_{\mathrm{machine}}$", color=REFERENCE_COLOR, fontsize=9,
            ha="right", va="top")
    ax.set_xlabel("step k")
    ax.set_title(title)
    style_axes(ax, xticks=range(0, n_spread + 1, 2))
axes[0].set_ylabel(r"$\|I - V_k^\mathsf{T} V_k\|_2$")
axes[0].set_ylim(1e-17, 10)
axes[0].legend(frameon=False, loc="upper left")
add_figure_caption(fig, "2.1", r"$\|I - V_k^\mathsf{T} V_k\|_2$ for Arnoldi with classical Gram–Schmidt (one pass), modified Gram–Schmidt and classical Gram–Schmidt applied twice, on diag_spread with eigenvalues over 4 decades (left) and 12 decades (right).")
plt.show()

**Reading Figure 2.1.** With one pass, orthogonality is lost, and the loss grows with every step.

- **4 decades (left):** both one-pass methods start at round-off and grow steadily, to $6.6 \times 10^{-12}$ (classical) and $5.0 \times 10^{-12}$ (modified) at $k = 20$. That much is harmless: to ten digits $\operatorname{cond}(V_{20})$ still prints as 1.
- **12 decades (right):** the same growth, much faster. Classical Gram–Schmidt is at $1.5 \times 10^{-9}$ by $k = 12$, $6.4 \times 10^{-7}$ by $k = 16$ and $1.7 \times 10^{-3}$ by $k = 18$, and at $k = 20$ the loss is $1.1$: the basis is not orthonormal in any useful sense, and $\operatorname{cond}(V_{20}) = 14$. Modified Gram–Schmidt loses less ($1.4 \times 10^{-10}$, $2.9 \times 10^{-8}$, $3.9 \times 10^{-7}$, then $5.4 \times 10^{-6}$ at $k = 20$), but it loses it the same way.
- **Two passes:** on both matrices, $\|I - V_k^\mathsf{T} V_k\|_2$ stays at about $3.6 \times 10^{-16}$ for every $k$, a couple of $\varepsilon_{\text{machine}}$, and $\operatorname{cond}(V_k)$ prints as 1.0000000000.

Why does one pass lose more at every step, and why does a second pass stop that? The table below follows each method's own run on the 12-decade matrix and prints, at a few steps $j$ (in the notes' 1-based numbering, so step $j$ applies $A$ to $v_j$ and makes $v_{j+1}$):

- the **loss of the basis so far**, $\|I - V_j^\mathsf{T} V_j\|_2$, before step $j$ adds a vector;
- the **error of the new vector**, $\max_{i \le j} |v_i^\mathsf{T} v_{j+1}|$, how far $v_{j+1}$ is from perpendicular to the earlier vectors;
- the **cancellation**, $\|A v_j\|_2 / h_{j+1,j}$: how many times shorter the remainder is than the $A v_j$ it came from, which says how much of $A v_j$ was removed.

The last line takes the clean basis from two passes and, at every step, orthogonalises $A v_j$ against it once and then once more, to measure what each pass leaves when it starts from an orthonormal basis.

In [ ]:
# Along each method's own run on the 12-decade matrix (A used for checking only).
for label, (V, H) in runs_12.items():
    print(label)
    print("   j   loss of V_j   error of v_{j+1}   cancellation ||A v_j|| / h_{j+1,j}")
    for j in (4, 8, 12, 16, 18):
        V_j = V[:, :j]                                     # v_1, ..., v_j
        loss = np.linalg.norm(np.eye(j) - V_j.T @ V_j, 2)
        new_error = np.max(np.abs(V_j.T @ V[:, j]))        # max over i <= j of |v_i^T v_{j+1}|
        cancellation = np.linalg.norm(A_spread_12 @ V[:, j - 1]) / H[j, j - 1]
        print(f"  {j:2d}   {loss:11.1e}   {new_error:16.1e}   {cancellation:12.3g}")
    print()

# Each pass on its own, starting from the clean basis of two passes.
V, _ = runs_12["classical Gram–Schmidt applied twice"]
after_one_pass, after_two_passes = [], []
for j in range(1, n_spread):
    basis = V[:, :j]
    w = A_spread_12 @ V[:, j - 1]
    w = w - basis @ (basis.T @ w)
    after_one_pass.append(np.max(np.abs(basis.T @ w)) / np.linalg.norm(w))
    w = w - basis @ (basis.T @ w)
    after_two_passes.append(np.max(np.abs(basis.T @ w)) / np.linalg.norm(w))
print("from a clean basis, largest error of the new vector over all steps:  "
      f"one pass {max(after_one_pass):.1e},  two passes {max(after_two_passes):.1e}")

**What one pass does with a basis that is already off.** Write $V = V_j$ for the basis so far and $w = A v_j$. One classical pass computes $w' = w - V (V^\mathsf{T} w)$, so

$$
V^\mathsf{T} w' = V^\mathsf{T} w - (V^\mathsf{T} V)(V^\mathsf{T} w) = (I - V^\mathsf{T} V)\, V^\mathsf{T} w .
$$

For an orthonormal $V$ this is zero. Otherwise it is up to about $\|I - V^\mathsf{T} V\|_2 \cdot \|w\|_2$, plus the pass's own rounding, about $\varepsilon_{\text{machine}} \|w\|_2$. The new vector is $w'$ divided by its length $h_{j+1,j}$, so its error is up to about

$$
\big( \|I - V_j^\mathsf{T} V_j\|_2 + \varepsilon_{\text{machine}} \big) \cdot \frac{\|A v_j\|_2}{h_{j+1,j}} :
$$

the basis's existing loss plus $\varepsilon_{\text{machine}}$, times the cancellation. That is what the table shows.

- **One pass compounds.** The cancellation is about 6.6 at every step, so each new vector comes out with a few times the loss the basis already had, and then joins the basis for the next step. Along classical Gram–Schmidt's run the loss grows by a factor of 4 to 5 per step: $7.2 \times 10^{-15}$ at $j = 4$, $4.1 \times 10^{-12}$ at $j = 8$, $1.5 \times 10^{-9}$ at $j = 12$. Modified Gram–Schmidt grows by about 4 per step from a smaller start. That growth is **inherited** error, not fresh round-off: from a clean basis, the last line shows, one pass leaves at most $1.8 \times 10^{-15}$, about $\varepsilon_{\text{machine}}$ times the cancellation. Near the end the basis is so far off that the cancellation itself grows (265 at $j = 16$ and $7 \times 10^4$ at $j = 18$ along classical Gram–Schmidt's run), and the loss runs away to order 1.
- **Two passes don't.** The second pass works on $w'$, which is already within about $10^{-15}$ (relative to its length) of perpendicular to the basis. There is almost nothing left to remove, so it cancels nothing, and it leaves an error of about $\varepsilon_{\text{machine}}$: at most $6.4 \times 10^{-17}$ in the last line. Each step therefore starts from a basis that is orthonormal to round-off, and the loss never compounds: $3.6 \times 10^{-16}$ at $j = 18$. A third pass would have nothing to do. This is the result known as "twice is enough", due to Kahan and published by Parlett (*The Symmetric Eigenvalue Problem*). It holds as long as $A v_j$ is not numerically inside the span of the basis already, which is the breakdown case.

**What the tutorial uses.** From here on the tutorial orthogonalises with classical Gram–Schmidt applied twice, the default of `arnoldi`, as production does. It does twice the arithmetic of one pass, and keeps the basis orthonormal to round-off. Two later steps depend on that. Step 3's smaller least-squares problem needs $V_{k+1}$ orthonormal. Step 6's breakdown test compares the remainder with $\varepsilon_{\text{machine}} \|A v_j\|_2$, which only makes sense if the orthogonalisation itself leaves no more than about $\varepsilon_{\text{machine}}$ behind. Step 9 times the two classical passes against modified Gram–Schmidt's loop, which is the other reason production chose them.

**Ties to code.** Production's `arnoldi_step(operator, basis, step)` in [arnoldi.py](../WhistlerRegulatedHeatDispersion/src/time_integration/implicit/arnoldi.py) *is* `arnoldi_step_classical_gram_schmidt_twice`. `project(basis, image)` is `V.T @ w`, `combine(basis, ...)` is `V @ components`, it does both twice, and `first_pass + second_pass` is the accumulated `h`. It returns the new Hessenberg column and a breakdown flag, with the same test as `normalise_remainder`. It differs in three ways, all because it is written in JAX (Step 9):

- it stores basis vectors as rows rather than columns;
- it returns a new basis instead of filling one in place, because JAX arrays can't be modified;
- it multiplies the whole preallocated basis, unused zero rows included, because JAX needs array shapes fixed in advance. The zero rows contribute nothing.

### 2.5 `gmres_v1`: Step 1's solver with the new basis

Write the iterate in the new basis, $x_k = V_k\, y$, with a weight vector $y \in \mathbb{R}^k$ that plays the role $c$ played in Step 1. The GMRES rule becomes

$$
\min_{y \in \mathbb{R}^k} \big\| b - (A V_k)\, y \big\|_2 ,
$$

the same least-squares problem as in Step 1 with $K_k$ replaced by $V_k$. The columns of $A V_k$ are $A v_1, \dots, A v_k$, exactly the vectors $w = A v_j$ that each Arnoldi step computes before it starts subtracting. `gmres_v1` keeps a copy of each, so it still spends **one matvec per step**.

Compared with `gmres_v0`, only two things change:

- the next basis vector is $A v_k$ with its components along $v_1, \dots, v_k$ removed (twice, as section 2.4 found necessary) and scaled to length 1, instead of $A^k b$;
- $c$ is renamed $y$.

Everything else stays: the $n \times k$ `lstsq` solve at every step, the free residual $r_k = b - (A V_k)\, y$, the stopping test and the `record`. Notice that `gmres_v1` computes the $h_{ij}$ (as `V.T @ w`) and then throws them away. Step 3 puts them to use.

`gmres_v1` writes the orthogonalisation inline instead of calling `arnoldi_step_classical_gram_schmidt_twice`. That function does its own matvec and doesn't return $A v_k$, which `gmres_v1` needs for its least-squares problem on $A V_k$. Calling it would cost a second matvec per step, or need the Arnoldi relation, which is Step 3.

There is one addition. If the remainder $w$ comes out negligible, there is no new direction to add, so the solver stops after this step. "Negligible" is the test of section 2.4, $\|w\|_2 \le \varepsilon_{\text{machine}} \cdot \|A v_k\|_2$, written as `not (‖w‖ > ε‖A v_k‖)` so that a NaN also counts. Step 6 explains both choices.

In [ ]:
def gmres_v1(matvec, b, *, rtol=1e-10, max_iterations=None, verbose=False):
    """GMRES over an orthonormal Krylov basis, built by Arnoldi with classical Gram-Schmidt applied twice.

    The same iterates as gmres_v0 in exact arithmetic. It still solves the
    n x k least-squares problem min_y ||b - (A V_k) y|| from scratch every step.

    Args:
        matvec: function v -> A v. The solver never sees A itself.
        b: right-hand side, shape (n,).
        rtol: stop once ||r_k|| <= rtol * ||b||.
        max_iterations: largest k to try. Defaults to n.
        verbose: print y and the residual at every step.

    Returns:
        x: the last iterate x_k.
        record: dict with
            "iterations"        number of steps k taken,
            "matvecs"           number of matvec calls,
            "residual_history"  [||r_0||, ||r_1||, ..., ||r_k||],
            "converged"         whether the tolerance was reached.
    """
    n = b.shape[0]
    if max_iterations is None:
        max_iterations = n
    b_norm = np.linalg.norm(b)

    # ----- k = 0: start from x_0 = 0, so the residual is b itself. -----
    x = np.zeros_like(b)
    residual_history = [b_norm]
    matvec_count = 0
    converged = False

    basis_vectors = []    # the columns of V_k:   v_1,   v_2,   v_3, ...
    A_basis_vectors = []  # the columns of A V_k: A v_1, A v_2, A v_3, ...
    next_basis_vector = b / b_norm  # v_1

    for k in range(1, max_iterations + 1):
        # 1. Grow the basis by one column. This is the step's only matvec.
        basis_vectors.append(next_basis_vector)
        A_basis_vectors.append(matvec(next_basis_vector))
        matvec_count += 1
        V = np.column_stack(basis_vectors)     # n x k
        AV = np.column_stack(A_basis_vectors)  # n x k

        # The next basis vector: A v_k minus its components along v_1, ..., v_k, removed
        # with classical Gram-Schmidt applied twice (2.4), then scaled to length 1.
        w = AV[:, -1]
        for _ in range(2):
            w = w - V @ (V.T @ w)
        norm_before = np.linalg.norm(AV[:, -1])
        breakdown = not (np.linalg.norm(w) > np.finfo(float).eps * norm_before)
        if not breakdown:
            next_basis_vector = w / np.linalg.norm(w)

        # 2. Find the weights y that make ||b - AV y|| as small as possible.
        y, *_ = np.linalg.lstsq(AV, b, rcond=None)

        # 3. The guess, and its residual (free: we already have A V).
        x = V @ y
        r = b - AV @ y
        residual_history.append(np.linalg.norm(r))

        if verbose:
            print(f"k = {k}:  y = {y},  ||r_k|| = {residual_history[-1]:.4e}")

        # 4. Stop once the residual is small enough.
        if residual_history[-1] <= rtol * b_norm:
            converged = True
            break
        # 5. No new direction: the Krylov space has stopped growing (Step 6).
        if breakdown:
            break

    record = {
        "iterations": k,
        "matvecs": matvec_count,
        "residual_history": residual_history,
        "converged": converged,
    }
    return x, record

**Checking "same math".** In exact arithmetic, `gmres_v0` and `gmres_v1` produce the same $x_k$ at every $k$. In floating point we expect agreement up to round-off, *as long as `gmres_v0` itself still works*.

`compare_versions(old, new, A, b)` from `tutorial_utils.py` runs both solvers for exactly $k$ steps, for $k = 1, 2, \dots$ (it passes `max_iterations=k` and `rtol=0`, so neither stops early). For each $k$ it reports:

- `x_difference`: $\|x_k^{\text{old}} - x_k^{\text{new}}\|_2 \,/\, \|x_k^{\text{new}}\|_2$, the relative difference between the two iterates;
- `residual_difference`: the difference between the two reported residual norms, divided by $\|b\|_2$;
- `matvecs_equal`: whether both spent the same number of matvecs.

First, `geo3` and Step 1's 100×100 `diag_narrow`, where `gmres_v0` worked. On `diag_narrow` we compare the 11 steps it needed in Step 1.

In [ ]:
x, record = gmres_v1(CountingMatvec(A_geo3), b_geo3, verbose=True)
print("x =", x, "   matvecs:", record["matvecs"])

for name, A, b, ks in [("geo3", A_geo3, b_geo3, None),
                       ("diag_narrow", A_narrow, b_narrow, range(1, 12))]:
    diff = compare_versions(gmres_v0, gmres_v1, A, b, ks=ks)
    print(f"\n{name}: gmres_v0 vs gmres_v1")
    print("   k   x_difference   residual_difference")
    for k, dx, dr in zip(diff["k"], diff["x_difference"], diff["residual_difference"]):
        print(f"  {k:2d}   {dx:12.1e}   {dr:19.1e}")
    assert diff["matvecs_equal"].all()
    assert diff["x_difference"].max() < 1e-11 and diff["residual_difference"].max() < 1e-11

On both matrices the two versions agree to round-off at every step, and spend the same number of matvecs. The largest difference between the iterates is $1.9 \times 10^{-15}$ on `geo3` and $5.8 \times 10^{-13}$ on `diag_narrow`. On `diag_narrow` it grows slowly with $k$, because $\operatorname{cond}(K_k)$ does, but it never gets near the $10^{-11}$ the check allows.

The weights are different, though. At $k = 1$, Step 1 found $c = 0.2941$, and here $y = 0.5094$. Both describe the same $x_1$: $x_1 = c\, b = y\, v_1 = y\, b / \beta$, so $y = \beta c = \sqrt 3 \times 0.2941 = 0.5094$. **Different basis, different weights, same iterate.**

Now the matrix where `gmres_v0` failed:

In [ ]:
x_v1, record_v1 = gmres_v1(CountingMatvec(A_spread), b_spread)
relative_v1 = np.array(record_v1["residual_history"]) / np.linalg.norm(b_spread)

print("converged:", record_v1["converged"], "  iterations:", record_v1["iterations"],
      "  matvecs:", record_v1["matvecs"])
print("||r_k|| / ||b|| for k = 0, 1, ..., 20:")
print(" ".join(f"{residual:.1e}" for residual in relative_v1))
print("\nsteps where the residual went UP:",
      [k for k in range(1, len(relative_v1)) if relative_v1[k] > relative_v1[k - 1]])

x_exact_spread = np.linalg.solve(A_spread, b_spread)  # checking only
print("largest error |x - x_exact|:", np.max(np.abs(x_v1 - x_exact_spread)))

Both predictions from Step 1 now hold. The residual never goes up, and at $k = 20 = n$ it reaches $2.8 \times 10^{-13}$, below the tolerance, so the solve converges. The largest error in $x$ is $4 \times 10^{-13}$. It needs all 20 steps, because the eigenvalues are spread over four decades (Step 5 explains why that makes GMRES slow), but every step is now computed correctly.

How closely do the two versions agree on this matrix? In Figure 2.2 below, the left panel overlays their residuals. The right panel shows the relative difference between their iterates at each $k$, against $\operatorname{cond}(K_k) \cdot \varepsilon_{\text{machine}}$. That product is roughly the relative error a least-squares solve with $K_k$ can pick up: Step 1 noted that a condition number $\kappa$ costs about $\log_{10} \kappa$ digits.

In [ ]:
diff_spread = compare_versions(gmres_v0, gmres_v1, A_spread, b_spread)
n_spread = b_spread.shape[0]
b_norm_spread = np.linalg.norm(b_spread)
eps = np.finfo(float).eps

K_spread = krylov_matrix(CountingMatvec(A_spread), b_spread, n_spread)
cond_K = np.array([np.linalg.cond(K_spread[:, :k]) for k in range(1, n_spread + 1)])

print("   k   cond(K_k)   cond(K_k)*eps   x_difference")
for k in range(1, 7):
    print(f"  {k:2d}   {cond_K[k - 1]:9.1e}   {cond_K[k - 1] * eps:13.1e}   {diff_spread['x_difference'][k - 1]:12.1e}")

fig, (ax_residual, ax_difference) = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)

plot_residuals(ax_residual, record_v0["residual_history"], b_norm=b_norm_spread,
               label="gmres_v0 (Krylov basis)", color=SERIES_COLORS[1])
plot_residuals(ax_residual, record_v1["residual_history"], b_norm=b_norm_spread,
               label="gmres_v1 (Arnoldi basis)", color=SERIES_COLORS[0])
ax_residual.set_title("residual on diag_spread (n = 20)")
ax_residual.legend(frameon=False, loc="lower left")

ax_difference.semilogy(diff_spread["k"], diff_spread["x_difference"], color=LINE_COLOR,
                       linewidth=2, marker="o", markersize=5,
                       label=r"$\|x_k^{v0} - x_k^{v1}\|_2 \,/\, \|x_k^{v1}\|_2$")
ax_difference.semilogy(diff_spread["k"], cond_K * eps, color=REFERENCE_COLOR, linewidth=1,
                       linestyle="--", label=r"cond$(K_k)\cdot\varepsilon_{\mathrm{machine}}$")
ax_difference.set_ylim(1e-17, 1e3)
ax_difference.set_xlabel("step k")
ax_difference.set_ylabel("relative difference")
ax_difference.set_title("how far apart the two iterates are")
ax_difference.legend(frameon=False, loc="lower right")

for ax in (ax_residual, ax_difference):
    style_axes(ax, xticks=range(0, n_spread + 1, 2))

add_figure_caption(fig, "2.2", 'gmres_v0 vs gmres_v1 on diag_spread. Left: their residuals. Right: the relative difference between their iterates, against cond$(K_k)\\cdot\\varepsilon_{\\mathrm{machine}}$.')
plt.show()

The table and the right panel of Figure 2.2 show three regimes.

- **$k \le 2$:** the iterates agree to $3 \times 10^{-16}$, full double precision.
- **$k = 3$ and $4$:** they differ by $3.6 \times 10^{-12}$ and then $1.1 \times 10^{-7}$. That is `gmres_v0`'s round-off, and it stays below the dashed line ($6.5 \times 10^{-9}$ and $6.6 \times 10^{-5}$).
- **$k \ge 5$:** $\operatorname{cond}(K_5) = 3.1 \times 10^{15}$, essentially $1/\varepsilon_{\text{machine}}$. The difference jumps to order 1 and stays there. `gmres_v0`'s iterates are wrong from here on, and `gmres_v1`'s are right: its residual (left panel) keeps falling.

So the label "same iterates up to round-off, wherever the previous version still works" has a definite meaning here. The iterates agree to within about $\operatorname{cond}(K_k) \cdot \varepsilon_{\text{machine}}$, and that bound is small only while $\operatorname{cond}(K_k)$ is.

### Summary of Step 2

- Any basis of $\mathcal{K}_k$ gives the same GMRES iterate in exact arithmetic. The basis only decides how much round-off the computation picks up.
- The **Arnoldi process** builds an orthonormal basis $v_1, v_2, \dots$ of the Krylov space, one vector per matvec. Each step applies $A$ to the newest $v_j$, subtracts its components $h_{ij}$ along the earlier vectors, does that subtraction a second time (classical Gram–Schmidt applied twice), and normalises what is left.
- `arnoldi_step_classical_gram_schmidt_twice` is one Arnoldi step as a function. It keeps the numbers it computes as one column of the $(k+1) \times k$ matrix $\bar H_k$. Two one-pass variants with the same arguments, `arnoldi_step_classical_gram_schmidt` and `arnoldi_step_modified_gram_schmidt`, are there for comparison.
- In floating point one pass loses orthogonality, and the loss compounds: each step inherits the basis's loss and multiplies it by the cancellation. By $k = 20$ it is $6.6 \times 10^{-12}$ (classical) and $5.0 \times 10^{-12}$ (modified) on `diag_spread`, and $1.1$ and $5.4 \times 10^{-6}$ when the eigenvalues spread over 12 decades. Two passes keep $\|I - V_k^\mathsf{T} V_k\|_2$ at about $4 \times 10^{-16}$ on both.
- **`gmres_v1`** is `gmres_v0` with that basis. On `diag_spread` its residual falls at every step, to $2.8 \times 10^{-13}$ at $k = 20$, where `gmres_v0` stalled at 0.95. Where `gmres_v0` still works, the two agree to within about $\operatorname{cond}(K_k) \cdot \varepsilon_{\text{machine}}$.

**Ties to code.** `arnoldi_step` in [arnoldi.py](../WhistlerRegulatedHeatDispersion/src/time_integration/implicit/arnoldi.py) is `arnoldi_step_classical_gram_schmidt_twice`, apart from what JAX requires (basis vectors as rows, a new basis returned instead of one filled in place, products with the whole preallocated basis), which is Step 9.

**Next: Step 3.** `gmres_v1` still solves an $n \times k$ least-squares problem from scratch at every step, keeps all of $A V_k$ in memory, and throws the $h_{ij}$ away. Step 3 shows that the $h_{ij}$ record each $A v_j$ in the new basis, $A V_k = V_{k+1} \bar H_k$, and that this shrinks the problem to $k+1$ equations in $k$ unknowns, with no $n$ in it at all.